# Nexus cycle attribution (v2)

Logic lives in `src/v2/scripts/` — this notebook is the driver.

Constants: `scripts/config.toml`.


In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# make scripts importable when the kernel cwd is anywhere under the repo
SCRIPTS = Path('/home/ajain/apeksha/pnl_attribution/src/v2/scripts')
ROOT = SCRIPTS.parent  # src/v2
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import CONFIG, get_ch_client
from scripts.day_attribution import attribute_all_books_day, attribute_book_day

client = get_ch_client()
print('books =', len(CONFIG.book_names))
print('contract size =', CONFIG.contract.size, '| clearing =', CONFIG.contract.clearing_rate)


books = 10
contract size = 1000 | clearing = 0.0019


### Single book / day

Columns include `abs_lots` plus `$` and `c/bbl` for each stage/component.
`c/bbl = $ / (abs_lots × 1000) × 100`.

In [2]:
DATE = date(2026, 9, 17)
BOOK = 'Hedger Spreads'

book_day = attribute_book_day(client, DATE, BOOK)

  Hedger Spreads: prefetching market data (476 cycles)…
  Hedger Spreads: 50/476 cycles
  Hedger Spreads: 100/476 cycles
  Hedger Spreads: 150/476 cycles
  Hedger Spreads: 200/476 cycles
  Hedger Spreads: 250/476 cycles
  Hedger Spreads: 300/476 cycles
  Hedger Spreads: 350/476 cycles
  Hedger Spreads: 400/476 cycles
  Hedger Spreads: 450/476 cycles


In [4]:
book_day

unit                                     meta            $                \
stage                                abs_lots        TOTAL                 
component                                     in_exec_cost out_exec_cost   
e61b65dc-d67c-4b85-9076-5ec383a55a67     55.0       4129.6        2951.3   
52e4df80-44da-4b3f-ac9e-201104366b48     32.0       3782.7        2412.3   
76b11f0d-d245-4875-853d-655653522efb     18.0       3072.6        1888.9   
15de9f1f-f7fe-487d-a21c-0580187ebc3d      6.0       2157.3        1175.5   
c7f68155-6c28-4825-a6b0-d3bdf6aa65ea    117.0       1957.2        1009.9   
...                                       ...          ...           ...   
19d28b05-0619-47fb-87b3-f0d838d6ac05     54.0       7290.0        4513.3   
ca5a10d2-8ba5-4218-9aa6-d9c0497450ea     41.0       6258.1        3747.8   
4f84a3e3-70f2-40de-952e-61cf0c49ee49     38.0       5625.0        3385.2   
7d2d1178-7db4-4472-8844-aa635a254b8b     41.0       5288.5        3248.0   
DAY TOTAL                             68517.0    3831509.9     2780195.9   

unit                                                                  \
stage                                                                  
component                            held_market_risk held_exec_save   
e61b65dc-d67c-4b85-9076-5ec383a55a67             0.00         1336.3   
52e4df80-44da-4b3f-ac9e-201104366b48             0.00         1429.4   
76b11f0d-d245-4875-853d-655653522efb          -198.68         1183.7   
15de9f1f-f7fe-487d-a21c-0580187ebc3d           128.62          981.8   
c7f68155-6c28-4825-a6b0-d3bdf6aa65ea            78.10          947.3   
...                                               ...            ...   
19d28b05-0619-47fb-87b3-f0d838d6ac05          -880.35         2824.3   
ca5a10d2-8ba5-4218-9aa6-d9c0497450ea          -816.24         2541.7   
4f84a3e3-70f2-40de-952e-61cf0c49ee49           -15.21         2287.4   
7d2d1178-7db4-4472-8844-aa635a254b8b           128.27         2040.5   
DAY TOTAL                                     5356.07      1147703.0   

unit                                                                        \
stage                                                        transfer_pred   
component                            value_added_by_strategy  in_exec_cost   
e61b65dc-d67c-4b85-9076-5ec383a55a67                 2514.60        1742.8   
52e4df80-44da-4b3f-ac9e-201104366b48                 2799.80        1691.2   
76b11f0d-d245-4875-853d-655653522efb                 2168.72        1397.9   
15de9f1f-f7fe-487d-a21c-0580187ebc3d                 2092.22        1023.2   
c7f68155-6c28-4825-a6b0-d3bdf6aa65ea                 1972.70         947.3   
...                                                      ...           ...   
19d28b05-0619-47fb-87b3-f0d838d6ac05                 4720.65        3349.3   
ca5a10d2-8ba5-4218-9aa6-d9c0497450ea                 4235.76        2918.2   
4f84a3e3-70f2-40de-952e-61cf0c49ee49                 4511.99        2617.0   
7d2d1178-7db4-4472-8844-aa635a254b8b                 4209.27        2438.4   
DAY TOTAL                                         2204373.07     1618243.3   

unit                                                                 \
stage                                                                 
component                            out_exec_cost held_market_risk   
e61b65dc-d67c-4b85-9076-5ec383a55a67        1742.8              0.0   
52e4df80-44da-4b3f-ac9e-201104366b48        1691.2              0.0   
76b11f0d-d245-4875-853d-655653522efb        1397.9              0.0   
15de9f1f-f7fe-487d-a21c-0580187ebc3d        1023.2              0.0   
c7f68155-6c28-4825-a6b0-d3bdf6aa65ea         947.3              0.0   
...                                            ...              ...   
19d28b05-0619-47fb-87b3-f0d838d6ac05        3349.3              0.0   
ca5a10d2-8ba5-4218-9aa6-d9c0497450ea        2918.2              0.0   
4f84a3e3-70f2-40de-952e-61cf0c49ee49        2617.0  

### All books / day

Index = `(book, cycle_id)` with per-book `DAY TOTAL` and a final `(TEAM, TOTAL)` row.

Columns = `(unit, stage, component)`:
- `meta.abs_lots` — Σ|triggered| lots this cycle (fallback: |to-be-hedged|)
- `$` / `c/bbl` — dollars and cents-per-barrel (`$ / (abs_lots × 1000) × 100`; totals are volume-weighted)

`value_added = in_exec − (out_exec − held_market_risk − held_exec_save)`

In [3]:
DATE = date(2026, 9, 17)

all_books_day = attribute_all_books_day(client, DATE)
all_books_day.round(2)


prefetching quote bars for 2026-09-17…
book=Hedger Spreads
  Hedger Spreads: prefetching curves (476 cycles)…
  Hedger Spreads: 50/476 cycles
  Hedger Spreads: 100/476 cycles
  Hedger Spreads: 150/476 cycles
  Hedger Spreads: 200/476 cycles
  Hedger Spreads: 250/476 cycles
  Hedger Spreads: 300/476 cycles
  Hedger Spreads: 350/476 cycles


KeyboardInterrupt: 

### Inspect one book slice


In [ ]:
# Hedger Spreads day total: abs_lots + $ and c/bbl for TOTAL stage
if not all_books_day.empty and 'Hedger Spreads' in all_books_day.index.get_level_values(0):
    day = all_books_day.loc[('Hedger Spreads', 'DAY TOTAL')]
    print('abs_lots =', float(day[('meta', 'abs_lots', '')]))
    display(day.xs('$', level='unit').xs('TOTAL', level='stage').round(2))
    display(day.xs('c/bbl', level='unit').xs('TOTAL', level='stage').round(4))